In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../india_weather_rainfall_data.xlsx")
df

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.0500,74.4000,0.1
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.0500,74.4000,4.4
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.0500,74.4000,2.3
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.0500,74.4000,35.0
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.0500,74.4000,25.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
970334,2025-02-06,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,27.3,22.0,34.0,7.6,1011.4,5,17.7167,83.2333,0.0
970335,2025-02-07,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.7,23.0,30.0,7.7,1013.1,5,17.7167,83.2333,0.3
970336,2025-02-08,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.5,23.0,30.0,8.7,1013.0,5,17.7167,83.2333,0.0
970337,2025-02-09,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.5,23.0,30.0,8.5,1012.8,5,17.7167,83.2333,0.0


In [3]:
# df = df.replace({np.nan: None})

In [4]:
df['date_of_record'] = df['date_of_record'].astype('str')

In [5]:
for col in df.select_dtypes(include=['datetime64[ns]']):
    df[col] = df[col].astype(str)

In [6]:
missing_values = {}
null_count = df.isnull().sum()
for n, i in enumerate(null_count):
    missing_values[df.columns[n]] = 100 - (df.shape[0] - i)/df.shape[0]*100

missing_values

{'date_of_record': 0.0,
 'month': 0.0,
 'season': 0.0,
 'station_name': 0.0,
 'state': 0.0,
 'district': 0.0,
 'avg_temp': 0.0,
 'min_temp': 4.523985947179284,
 'max_temp': 11.3978722899935,
 'wind_speed': 28.283311296361376,
 'air_pressure': 31.397686787813328,
 'elevation': 0.0,
 'latitude': 0.0,
 'longitude': 0.0,
 'rainfall': 26.542682505804677}

In [7]:
def district_groupwise_data(target):
    district_avg_temp_dict = df.groupby(['state','district'])[target].mean().reset_index().to_dict(orient='records')
    district_avg_temp = {}

    for i in district_avg_temp_dict:
        if i['state'] not in district_avg_temp:
            district_avg_temp[i['state']] = {}

        district_avg_temp[i['state']][i['district']] = i[target]

    return district_avg_temp

In [8]:
def station_groupwise_data(target):
    station_avg_temp_dict = df.groupby(['state','district','station_name'])[target].mean().reset_index().to_dict(orient='records')
    station_avg_temp = {}

    for i in station_avg_temp_dict:
        if i['state'] not in station_avg_temp:
            station_avg_temp[i['state']] = {}

        if i['district'] not in station_avg_temp[i['state']]:
            station_avg_temp[i['state']][i['district']] = {}

        station_avg_temp[i['state']][i['district']][i['station_name']] = i[target]

    return station_avg_temp

In [9]:
def season_groupwise_data(target):
    season_avg_temp_dict = df.groupby(['state','district','station_name', 'season'])[target].mean().reset_index().to_dict(orient='records')
    season_avg_temp = {}

    for i in season_avg_temp_dict:
        if i['state'] not in season_avg_temp:
            season_avg_temp[i['state']] = {}

        if i['district'] not in season_avg_temp[i['state']]:
            season_avg_temp[i['state']][i['district']] = {}

        if i['station_name'] not in season_avg_temp[i['state']][i['district']]:
                season_avg_temp[i['state']][i['district']][i['station_name']] = {}

        season_avg_temp[i['state']][i['district']][i['station_name']][i['season']] = i[target]

    return season_avg_temp

In [10]:
count = list(np.histogram(df['avg_temp'], bins=1000))


In [11]:
df = df.dropna(subset=['date_of_record'])
df['date_of_record'] = pd.to_datetime(df['date_of_record'])
df['max_temp'] = df['max_temp'].fillna(df.groupby(['month', 'station_name'])['max_temp'].transform('mean'))
df['max_temp'] = df['max_temp'].fillna(df['max_temp'].median())
df['min_temp'] = df['min_temp'].fillna(df.groupby(['month', 'station_name'])['min_temp'].transform('mean'))
df['min_temp'] = df['min_temp'].fillna(df['min_temp'].median())
df['wind_speed'] = df['wind_speed'].fillna(df.groupby(['month', 'station_name'])['wind_speed'].transform('mean'))
df['wind_speed'] = df['wind_speed'].fillna(df['wind_speed'].median())
df['air_pressure'] = df['air_pressure'].fillna(df.groupby(['month', 'station_name'])['air_pressure'].transform('mean'))
df['air_pressure'] = df['air_pressure'].fillna(df['air_pressure'].median())
df['rainfall'] = df['rainfall'].fillna(df.groupby(['month', 'station_name'])['rainfall'].transform('mean'))
df['rainfall'] = df['rainfall'].fillna(df['rainfall'].median())

df = df.drop_duplicates()

In [12]:
temp_series = df.groupby('date_of_record')['avg_temp'].mean()
min_temp_series = df.groupby('date_of_record')['min_temp'].mean()
max_temp_series = df.groupby('date_of_record')['max_temp'].mean()
wind_speed_series = df.groupby('date_of_record')['wind_speed'].mean()
air_pressure_series = df.groupby('date_of_record')['air_pressure'].mean()
rainfall_series = df.groupby('date_of_record')['rainfall'].mean()

In [13]:
df_sample = df.sample(10000)
df_sample['date_of_record'] = df_sample['date_of_record'].astype('str')

In [14]:
df_state_sample = df.groupby('state', group_keys=False)\
    .apply(lambda x: x.sample(min(len(x), 500)))\
    .reset_index(drop=True)

df_season_sample = df.groupby('season', group_keys=False).apply(lambda x: x.sample(min(len(x), 500))).reset_index(drop=True)
df_state_sample['date_of_record'] = df_state_sample['date_of_record'].astype(str)
df_season_sample['date_of_record'] = df_season_sample['date_of_record'].astype(str)

/var/folders/7c/lq4bm735423dshfysxglzw080000gn/T/ipykernel_41577/2441362625.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 500)))\
/var/folders/7c/lq4bm735423dshfysxglzw080000gn/T/ipykernel_41577/2441362625.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_season_sample = df.groupby('season', group_keys=False).apply(lambda x: x.sample(min(len(x), 500))).reset_index(drop

In [15]:
def outlier_stats(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "q1": float(q1),
        "q3": float(q3),
        "median": float(series.median()),
        "lower": float(lower),
        "upper": float(upper),
        "outliers": int(((series < lower) | (series > upper)).sum())
    }


In [16]:
corr = df.corr(numeric_only=True)

In [17]:
target_corr = df.corr(numeric_only=True)['avg_temp'].drop('avg_temp').sort_values(ascending=False)
target_corr

min_temp        0.917605
max_temp        0.916885
wind_speed      0.300535
rainfall        0.028063
longitude      -0.054895
latitude       -0.296831
elevation      -0.439126
air_pressure   -0.703504
Name: avg_temp, dtype: float64

In [18]:
monthly_trend = df.groupby('month').agg({
    'avg_temp': 'mean',
    'max_temp': 'mean',
    'min_temp': 'mean',
    'wind_speed': 'mean',
    'air_pressure': 'mean',
    'rainfall': 'mean',
}).reset_index(drop=False)
month_map = {
    'January': 1,
    'February': 2,
    'March': 3,
    'April': 4,
    'May': 5,
    'June': 6,
    'July': 7,
    'August': 8,
    'September': 9,
    'October': 10,
    'November': 11,
    'December': 12,
}
monthly_trend['weight'] = monthly_trend['month'].map(month_map)
monthly_trend = monthly_trend.sort_values(by='weight')
monthly_trend = monthly_trend.drop(columns=['weight'])
monthly_trend

,month,avg_temp,max_temp,min_temp,wind_speed,air_pressure,rainfall
4,January,19.594499,25.952087,13.943598,8.122101,1014.790235,1.014606
3,February,22.235994,28.973097,15.800750,8.671941,1013.847241,0.660825
7,March,25.979654,32.661373,19.515909,8.973912,1011.397551,1.137699
0,April,29.100049,35.574809,22.894483,9.718318,1008.588867,1.726433
8,May,30.315485,36.115868,24.791723,10.894192,1005.709215,4.080872
6,June,29.544730,34.562546,25.246434,11.772121,1003.815032,8.087275
5,July,27.781627,31.694675,24.610366,12.073179,1003.260618,12.317639
1,August,27.437673,31.355166,24.234667,11.036327,1004.956750,9.633181
11,September,27.236874,31.508012,23.719059,9.563714,1006.333182,8.752536
10,October,26.124751,31.468095,21.415806,7.360877,1010.517785,4.822965


In [19]:
stat = df.describe().reset_index().to_dict(orient='records')
for idx, row in enumerate(stat):
    for k, v in row.items():
        if isinstance(v, pd.Timestamp):
            stat[idx][k] = str(v)
        if pd.isna(v):
            stat[idx][k] = None

In [20]:
kpi_data = {}

state_values = df.groupby(['state']).agg({
    'avg_temp' : 'mean',
    'min_temp' : 'mean',
    'max_temp' : 'mean',
    'wind_speed' : 'mean',
    'air_pressure' : 'mean',
    'rainfall' : 'mean',
}).reset_index().to_dict(orient='records')

for i in state_values:
    kpi_data[i['state']] = {
        'avg_temp': i['avg_temp'],
        'min_temp': i['min_temp'],
        'max_temp': i['max_temp'],
        'wind_speed': i['wind_speed'],
        'air_pressure': i['air_pressure'],
        'rainfall': i['rainfall'],
        'district': {}
    }

district_values = df.groupby(['state', 'district']).agg({
    'avg_temp' : 'mean',
    'min_temp' : 'mean',
    'max_temp' : 'mean',
    'wind_speed' : 'mean',
    'air_pressure' : 'mean',
    'rainfall' : 'mean',
}).reset_index().to_dict(orient='records')

for i in district_values:
    kpi_data[i['state']]['district'][i['district']] = {
        'avg_temp': i['avg_temp'],
        'min_temp': i['min_temp'],
        'max_temp': i['max_temp'],
        'wind_speed': i['wind_speed'],
        'air_pressure': i['air_pressure'],
        'rainfall': i['rainfall'],
        'station_name': {}
    }

station_values = df.groupby(['state', 'district', 'station_name']).agg({
    'avg_temp' : 'mean',
    'min_temp' : 'mean',
    'max_temp' : 'mean',
    'wind_speed' : 'mean',
    'air_pressure' : 'mean',
    'rainfall' : 'mean',
}).reset_index().to_dict(orient='records')

for i in station_values:
    kpi_data[i['state']]['district'][i['district']]['station_name'][i['station_name']] = {
        'avg_temp': i['avg_temp'],
        'min_temp': i['min_temp'],
        'max_temp': i['max_temp'],
        'wind_speed': i['wind_speed'],
        'air_pressure': i['air_pressure'],
        'rainfall': i['rainfall'],
        'season': {}
    }

season_values = df.groupby(['state', 'district', 'station_name', 'season']).agg({
    'avg_temp' : 'mean',
    'min_temp' : 'mean',
    'max_temp' : 'mean',
    'wind_speed' : 'mean',
    'air_pressure' : 'mean',
    'rainfall' : 'mean',
}).reset_index().to_dict(orient='records')

for i in season_values:
    kpi_data[i['state']]['district'][i['district']]['station_name'][i['station_name']]['season'][i['season']] = {
        'avg_temp': i['avg_temp'],
        'min_temp': i['min_temp'],
        'max_temp': i['max_temp'],
        'wind_speed': i['wind_speed'],
        'air_pressure': i['air_pressure'],
        'rainfall': i['rainfall'],
    }

kpi_data

{'AN': {'avg_temp': 27.4089872486513,
  'min_temp': 25.1610276284994,
  'max_temp': 29.996525359599392,
  'wind_speed': 15.06648400353709,
  'air_pressure': 1009.5592079625159,
  'rainfall': 7.3986748322369085,
  'district': {'Nicobar': {'avg_temp': 27.506747366061422,
    'min_temp': 25.755189419412687,
    'max_temp': 29.264066352835687,
    'wind_speed': 18.032369423895986,
    'air_pressure': 1009.760031383098,
    'rainfall': 6.584263618022865,
    'station_name': {'Car Nicobar': {'avg_temp': 27.472158708809683,
      'min_temp': 25.51698049764627,
      'max_temp': 29.33883658372562,
      'wind_speed': 18.049731002017484,
      'air_pressure': 1009.7239744451917,
      'rainfall': 6.576529926025555,
      'season': {'Monsoon': {'avg_temp': 27.44888888888889,
        'min_temp': 25.990277777777777,
        'max_temp': 29.068472222222223,
        'wind_speed': 23.02611111111111,
        'air_pressure': 1008.7658333333334,
        'rainfall': 8.501249999999999},
       'Post-monsoo

In [21]:
data = {
    # dataset details
    "shape":df.shape,
    "columns": df.columns.tolist(),

    # unique values
    "season_values": df['season'].unique().tolist(),
    "station_count": df['station_name'].nunique(),
    # "state": df['state'].unique().tolist(),
    # "district": df.groupby('state')['district'].unique().apply(list).to_dict(),
    # "station": df.groupby('district')['station_name'].unique().apply(list).to_dict(),
    # "season": df.groupby('station_name')['season'].unique().apply(list).to_dict(),

    # missing value count
    "missing_values": missing_values,
    "null_count": null_count.to_dict(),

    "kpi_data": kpi_data,

    # mean values
    # "state_avg_temp": df.groupby('state')['avg_temp'].mean().to_dict(),
    # "district_avg_temp": district_groupwise_data('avg_temp'),
    # "station_avg_temp": station_groupwise_data('avg_temp'),
    # "season_avg_temp": season_groupwise_data('avg_temp'),
    # "state_min_temp": df.groupby('state')['min_temp'].mean().to_dict(),
    # "state_max_temp": df.groupby('state')['max_temp'].mean().to_dict(),
    # "state_wind_speed": df.groupby('state')['wind_speed'].mean().to_dict(),
    # "state_air_pressure": df.groupby('state')['air_pressure'].mean().to_dict(),
    # "state_rainfall": df.groupby('state')['rainfall'].mean().to_dict(),
    # "district_rainfall": district_groupwise_data('rainfall'),
    # "station_rainfall": station_groupwise_data('rainfall'),
    # "season_rainfall": season_groupwise_data('rainfall'),

    # time series trend analysis
    "avg_temp_time": [temp_series.index.astype(str).tolist(), temp_series.values.tolist()],
    "min_temp_time": [min_temp_series.index.astype(str).tolist(), min_temp_series.values.tolist()],
    "max_temp_time": [max_temp_series.index.astype(str).tolist(), max_temp_series.values.tolist()],
    "wind_speed_time": [wind_speed_series.index.astype(str).tolist(), wind_speed_series.values.tolist()],
    "air_pressure_time": [air_pressure_series.index.astype(str).tolist(), air_pressure_series.values.tolist()],
    "rainfall_time": [rainfall_series.index.astype(str).tolist(), rainfall_series.values.tolist()],

    # location analysis
    # "avg_temp_district": df.groupby('district')['avg_temp'].mean().to_dict(),
    # "avg_temp_station": df.groupby('station_name')['avg_temp'].mean().to_dict(),
    # "min_temp_district": df.groupby('district')['min_temp'].mean().to_dict(),
    # "min_temp_station": df.groupby('station_name')['min_temp'].mean().to_dict(),
    # "max_temp_district": df.groupby('district')['max_temp'].mean().to_dict(),
    # "max_temp_station": df.groupby('station_name')['max_temp'].mean().to_dict(),
    # "wind_speed_district": df.groupby('district')['wind_speed'].mean().to_dict(),
    # "wind_speed_station": df.groupby('station_name')['wind_speed'].mean().to_dict(),
    # "air_pressure_district": df.groupby('district')['air_pressure'].mean().to_dict(),
    # "air_pressure_station": df.groupby('station_name')['air_pressure'].mean().to_dict(),
    # "rainfall_district": df.groupby('district')['rainfall'].mean().to_dict(),
    # "rainfall_station": df.groupby('station_name')['rainfall'].mean().to_dict(),
    "avg_temp_lat_long" : df.groupby(['latitude','longitude']).agg({
        'avg_temp': 'mean',
        'min_temp': 'mean',
        'max_temp': 'mean',
        'wind_speed': 'mean',
        'air_pressure': 'mean',
        'rainfall': 'mean',
    }).reset_index().to_dict(orient='records'),
    "rainfall_lat_long": df.groupby(['latitude','longitude'])['rainfall'].mean().reset_index().to_dict(orient='records'),

    # distribution analysis
    "avg_temp_hist": {'count': np.histogram(df['avg_temp'], bins=1000)[0].tolist(), 'bins': np.histogram(df['avg_temp'], bins=1000)[1].tolist()},
    "min_temp_hist": {'count': np.histogram(df['min_temp'], bins=1000)[0].tolist(), 'bins': np.histogram(df['min_temp'], bins=1000)[1].tolist()},
    "max_temp_hist": {'count': np.histogram(df['max_temp'], bins=1000)[0].tolist(), 'bins': np.histogram(df['max_temp'], bins=1000)[1].tolist()},
    "wind_speed_hist": {'count': np.histogram(df['wind_speed'], bins=1000)[0].tolist(), 'bins': np.histogram(df['wind_speed'], bins=1000)[1].tolist()},
    "air_pressure_hist": {'count': np.histogram(df['air_pressure'], bins=1000)[0].tolist(), 'bins': np.histogram(df['air_pressure'], bins=1000)[1].tolist()},
    "rainfall_hist": {'count': np.histogram(df['rainfall'], bins=1000)[0].tolist(), 'bins': np.histogram(df['rainfall'], bins=1000)[1].tolist()},

    # outlier analysis
    "boxplot" : {
        "avg_temp": df_sample['avg_temp'].dropna().tolist(),
        "min_temp": df_sample['min_temp'].dropna().tolist(),
        "max_temp": df_sample['max_temp'].dropna().tolist(),
        "wind_speed": df_sample['wind_speed'].dropna().tolist(),
        "air_pressure": df_sample['air_pressure'].dropna().tolist(),
        "rainfall": df_sample['rainfall'].dropna().tolist(),
        "elevation": df_sample['elevation'].dropna().tolist(),
    },
    "outlier_stat": {
        "avg_temp": outlier_stats(df['avg_temp']),
        "min_temp": outlier_stats(df['min_temp']),
        "max_temp": outlier_stats(df['max_temp']),
        "wind_speed": outlier_stats(df['wind_speed']),
        "air_pressure": outlier_stats(df['air_pressure']),
        "rainfall": outlier_stats(df['rainfall'])
    },

    # scatter plot
    "scatter_sample": df_sample.to_dict(orient='records'),
    "scatter_sample_state": df_state_sample.to_dict(orient='records'),
    "scatter_sample_season": df_season_sample.to_dict(orient = 'records'),

    # correlation heatmap of numeric columns
    "corr": {
        'columns': corr.columns.tolist(),
        'values': corr.values.tolist()
    },

    # monthly trend
    "monthly_trend": monthly_trend.to_dict(orient='records'),

    # data stats
    'stats': stat,
}

In [23]:

with open('data.json', 'w') as f:
    json.dump(data, f, indent=4, default=str)

print("Data dumped successfully!")

Data dumped successfully!
